In [1]:
# %%
# ============================================================
# USA + WSJ Full Dataset Topic Classification
#
# Model:
# GPT-5-mini
#
# Final prompt:
# P3_EN
#
# Output:
# topic_primary_P3_EN
#
# ============================================================


# %%
# ============================================================
# 0. OpenAI setup
# ============================================================


from openai import OpenAI
from dotenv import load_dotenv

import os
import pandas as pd
import json
import time

from pathlib import Path


load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)


client = OpenAI()


MODEL = "gpt-5-mini"


print(
    "Model:",
    MODEL
)


Model: gpt-5-mini


In [2]:

# %%
# ============================================================
# 1. Paths
# ============================================================


BASE_DIR = Path(
    "/Users/yurujia/Desktop/Dissertation Data/USA"
)



USA_PATH = (

    BASE_DIR /

    "descriptive_stats_USA_Today/"

    "USA_Today_AV_final_with_first_av_relevant_paragraph.xlsx"

)



WSJ_PATH = (

    BASE_DIR /

    "descriptive_stats_WSJ/"

    "WSJ_AV_final_analytical_dataset.xlsx"

)



BATCH_INPUT_PATH = (

    BASE_DIR /

    "english_topic_P3_EN_batch_input.jsonl"

)



BATCH_OUTPUT_PATH = (

    BASE_DIR /

    "english_topic_P3_EN_batch_output.jsonl"

)



OUTPUT_PATH = (

    BASE_DIR /

    "english_AV_topic_final_P3_EN.xlsx"

)



print(
    USA_PATH
)

print(
    WSJ_PATH
)


/Users/yurujia/Desktop/Dissertation Data/USA/descriptive_stats_USA_Today/USA_Today_AV_final_with_first_av_relevant_paragraph.xlsx
/Users/yurujia/Desktop/Dissertation Data/USA/descriptive_stats_WSJ/WSJ_AV_final_analytical_dataset.xlsx


In [3]:

# %%
# ============================================================
# 2. Load datasets
# ============================================================


usa = pd.read_excel(
    USA_PATH
)



wsj = pd.read_excel(
    WSJ_PATH
)



print(
    "USA:",
    usa.shape
)


print(
    "WSJ:",
    wsj.shape
)


USA: (287, 87)
WSJ: (367, 57)


In [4]:
# %%
# ============================================================
# 3. Prepare topic dataframe
# ============================================================


usa_topic = pd.DataFrame({

    "id":
        range(len(usa)),

    "source":
        "USA Today",

    "text":
        usa[
            "first_av_relevant_paragraph"
        ]

})



wsj_topic = pd.DataFrame({

    "id":
        range(len(wsj)),

    "source":
        "WSJ",

    "text":
        wsj[
            "abstract"
        ]

})



# Important:
# avoid duplicate IDs after combining

usa_topic["id"] = range(
    len(usa_topic)
)


wsj_topic["id"] = range(
    len(usa_topic),
    len(usa_topic)+len(wsj_topic)
)



all_articles = pd.concat(
    [
        usa_topic,
        wsj_topic
    ],
    ignore_index=True
)



all_articles = (
    all_articles
    .dropna(
        subset=[
            "text"
        ]
    )
)



all_articles["text"] = (

    all_articles["text"]

    .astype(str)

    .str.strip()

)



all_articles = all_articles[
    all_articles["text"].str.len() > 0
]



all_articles = (
    all_articles
    .reset_index(drop=True)
)



print(
    "Articles for topic classification:",
    len(all_articles)
)



display(
    all_articles.head()
)



Articles for topic classification: 654


,id,source,text
0,0,USA Today,"Fiat Chrysler Automobiles revealed a new, semi..."
1,1,USA Today,Fiat Chrysler and Google already have a partne...
2,2,USA Today,The technology company confirmed that it's tak...
3,3,USA Today,"Tesla Motors CEO Elon Musk is letting 1,000 cu..."
4,4,USA Today,"Nvidia is at the intersection of key, changing..."


In [5]:

# %%
# ============================================================
# 4. Topic categories
# ============================================================


TOPICS = [

"Technology and Innovation",

"Safety and Risk",

"Policy and Regulation",

"Business and Commercialisation",

"Public Acceptance and Trust",

"Mobility and Social Impact",

"Environment and Sustainability",

"Legal and Ethics",

"Other"

]




In [6]:

# %%
# ============================================================
# 5. Final P3_EN Prompt
# ============================================================


def build_prompt_p3_en(text):

    return f"""

You are an expert media analyst specialising in autonomous vehicle (AV) news coverage.

Your task is to identify the SINGLE dominant topic (primary frame) of the following AV-related news article.

Select exactly ONE topic from the predefined categories below.

Do not create new categories.
Do not use alternative wording.
Return only the category name.


Available topics:


Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other



Topic definitions:


Technology and Innovation

Choose this topic when the article mainly focuses on:

- autonomous driving technology development
- AI systems, sensors, software, algorithms
- engineering progress
- technical capabilities
- testing performance
- technological improvements


Safety and Risk

Choose this topic when the article mainly focuses on:

- crashes
- failures
- safety concerns
- reliability problems
- operational risks
- safety evaluation


Policy and Regulation

Choose this topic when the article mainly focuses on:

- government policies
- legislation
- standards
- regulatory approval
- permits
- official rules or decisions


Business and Commercialisation

Choose this topic when the article mainly focuses on:

- company strategy
- investment
- acquisitions
- partnerships
- competition
- financial performance
- business models
- market development


Public Acceptance and Trust

Choose this topic when the article mainly focuses on:

- public opinion
- consumer attitudes
- trust
- concerns about adoption
- willingness to use autonomous vehicles


Mobility and Social Impact

Choose this topic when the article mainly focuses on:

- robotaxi services
- autonomous ride-hailing
- passenger transportation
- mobility solutions
- transportation system changes
- accessibility
- social impacts


Environment and Sustainability

Choose this topic when the article mainly focuses on:

- emissions
- energy efficiency
- environmental benefits
- sustainability


Legal and Ethics

Choose this topic when the article mainly focuses on:

- liability
- responsibility
- legal disputes
- ethical dilemmas
- accountability


Decision rules:


1.

Choose the topic representing the central framing of the article,
not all themes mentioned.


2.

Do not classify based on isolated keywords.


3.

When multiple topics appear, select the issue receiving the greatest emphasis.


4.

Distinguish between:

technical development
→ Technology and Innovation


company strategy or market activity
→ Business and Commercialisation


passenger services and transportation changes
→ Mobility and Social Impact


government rules and approvals
→ Policy and Regulation


accidents and failures
→ Safety and Risk


5.

Objective reporting does not automatically mean Business or Technology.

Classify according to the substantive issue discussed.


6.

If autonomous vehicles are only mentioned briefly without a clear thematic focus:

choose Other.



Return only ONE topic name.


Text:

{text}

""".strip()


In [7]:


# %%
# ============================================================
# 6. Create Batch input
# ============================================================


with open(
    BATCH_INPUT_PATH,
    "w",
    encoding="utf-8"
) as f:


    for idx,row in all_articles.iterrows():


        request = {


            "custom_id":

                f"article_{idx}",


            "method":

                "POST",


            "url":

                "/v1/responses",


            "body":{


                "model":

                    MODEL,


                "input":

                    build_prompt_p3_en(
                        row["text"]
                    )

            }

        }



        f.write(

            json.dumps(
                request,
                ensure_ascii=False
            )

            +

            "\n"

        )



print(
    "Batch created:"
)


print(
    BATCH_INPUT_PATH
)




Batch created:
/Users/yurujia/Desktop/Dissertation Data/USA/english_topic_P3_EN_batch_input.jsonl


In [8]:

# %%
# ============================================================
# 7. Upload batch
# ============================================================


batch_input = client.files.create(

    file=open(
        BATCH_INPUT_PATH,
        "rb"
    ),

    purpose="batch"

)



print(
    batch_input.id
)



file-Mwgks1gJJw2V9pewYxioxZ


In [9]:

# %%
# ============================================================
# 8. Create batch job
# ============================================================


batch_job = client.batches.create(

    input_file_id=batch_input.id,

    endpoint="/v1/responses",

    completion_window="24h"

)



print(
    batch_job.id
)


batch_6a670450073c8190bdda77903cf368d5


In [10]:

# %%
# ============================================================
# 9. Wait for completion
# ============================================================


while True:


    status = client.batches.retrieve(
        batch_job.id
    )


    print(
        status.status
    )


    if status.status == "completed":

        break


    if status.status in [
        "failed",
        "expired",
        "cancelled"
    ]:

        raise Exception(
            status.status
        )


    time.sleep(60)



validating
in_progress
in_progress
in_progress
in_progress
in_progress
finalizing
completed


In [11]:


# %%
# ============================================================
# 10. Download output
# ============================================================


output_file_id = (

    status.output_file_id

)



file_response = client.files.content(
    output_file_id
)



with open(

    BATCH_OUTPUT_PATH,

    "w",

    encoding="utf-8"

) as f:

    f.write(
        file_response.text
    )



print(
    BATCH_OUTPUT_PATH
)



# %%
# ============================================================
# 11. Parse outputs
# ============================================================


predictions = []

unexpected_outputs = []



with open(

    BATCH_OUTPUT_PATH,

    encoding="utf-8"

) as f:


    for line in f:


        item = json.loads(line)



        article_id = int(

            item["custom_id"]

            .replace(
                "article_",
                ""
            )

        )



        body = item["response"]["body"]


        output_text = None



        for output_item in body["output"]:


            if output_item.get("type")=="message":


                for content_item in output_item.get(
                    "content",
                    []
                ):


                    if content_item.get("type")=="output_text":


                        output_text = content_item["text"]



        if output_text is None:

            continue



        output_text = (
            output_text
            .strip()
        )



        if output_text in TOPICS:


            predictions.append({

                "id":
                    article_id,

                "topic_primary_P3_EN":
                    output_text

            })


        else:

            unexpected_outputs.append({

                "id":
                    article_id,

                "output":
                    output_text

            })



prediction_df = pd.DataFrame(
    predictions
)



print(
    "Parsed:",
    len(prediction_df)
)


print(
    "Unexpected:",
    len(unexpected_outputs)
)



# %%
# ============================================================
# 12. Merge
# ============================================================


final_topic_results = (

    all_articles

    .merge(

        prediction_df,

        on="id",

        how="left"

    )

)



final_topic_results["model"] = MODEL


final_topic_results["prompt"] = "P3_EN"


final_topic_results["date_processed"] = pd.Timestamp.now()



print(
    "Missing:",
    final_topic_results[
        "topic_primary_P3_EN"
    ]
    .isna()
    .sum()
)



display(
    final_topic_results.head()
)


/Users/yurujia/Desktop/Dissertation Data/USA/english_topic_P3_EN_batch_output.jsonl
Parsed: 654
Unexpected: 0
Missing: 0


,id,source,text,topic_primary_P3_EN,model,prompt,date_processed
0,0,USA Today,"Fiat Chrysler Automobiles revealed a new, semi...",Business and Commercialisation,gpt-5-mini,P3_EN,2026-07-27 15:17:35.088103
1,1,USA Today,Fiat Chrysler and Google already have a partne...,Business and Commercialisation,gpt-5-mini,P3_EN,2026-07-27 15:17:35.088103
2,2,USA Today,The technology company confirmed that it's tak...,Business and Commercialisation,gpt-5-mini,P3_EN,2026-07-27 15:17:35.088103
3,3,USA Today,"Tesla Motors CEO Elon Musk is letting 1,000 cu...",Technology and Innovation,gpt-5-mini,P3_EN,2026-07-27 15:17:35.088103
4,4,USA Today,"Nvidia is at the intersection of key, changing...",Business and Commercialisation,gpt-5-mini,P3_EN,2026-07-27 15:17:35.088103


In [12]:

# %%
# ============================================================
# 13. Topic distribution
# ============================================================


topic_distribution = (

    final_topic_results[
        "topic_primary_P3_EN"
    ]

    .value_counts()

)



display(
    topic_distribution
)



topic_percentage = (

    topic_distribution

    /

    topic_distribution.sum()

    *

    100

)



display(
    topic_percentage.round(2)
)



# %%
# ============================================================
# 14. Save
# ============================================================


final_topic_results.to_excel(

    OUTPUT_PATH,

    index=False

)



print(
    OUTPUT_PATH
)



# %%
# ============================================================
# 15. Save unexpected outputs
# ============================================================


if len(unexpected_outputs)>0:


    pd.DataFrame(
        unexpected_outputs
    ).to_excel(

        BASE_DIR /

        "english_topic_P3_EN_unexpected_outputs.xlsx",

        index=False

    )

topic_primary_P3_EN
Business and Commercialisation    319
Technology and Innovation          98
Safety and Risk                    67
Mobility and Social Impact         50
Policy and Regulation              44
Legal and Ethics                   37
Public Acceptance and Trust        21
Other                              17
Environment and Sustainability      1
Name: count, dtype: int64

topic_primary_P3_EN
Business and Commercialisation    48.78
Technology and Innovation         14.98
Safety and Risk                   10.24
Mobility and Social Impact         7.65
Policy and Regulation              6.73
Legal and Ethics                   5.66
Public Acceptance and Trust        3.21
Other                              2.60
Environment and Sustainability     0.15
Name: count, dtype: float64

/Users/yurujia/Desktop/Dissertation Data/USA/english_AV_topic_final_P3_EN.xlsx
